# 03 — Small-Molecule Binder Design

Targets (RFD3 paper §3.4): **FAD** (`7BKC`), **OQO** (`7V11`), **IAI** (`5SDV`), **SAM** (`7C7M`).

Per target:
- extract ligand HETATM record → mini PDB
- generate ~12 backbones around it (length 150)
- compute pocket-burial metrics

In [ ]:
%cd /content/repo
import sys
if '/content/repo/scripts' not in sys.path:
    sys.path.insert(0, '/content/repo/scripts')

from utils import (RESULTS, DATA, RunRecord, append_record, fetch_pdb,
                   free_gpu, load_ca_coords, rfd3_run)
import numpy as np, time, json, os
from pathlib import Path
import biotite.structure.io.pdb as bpdb

LIGANDS = {
    'FAD': ('7bkc', 'FAD'),
    'OQO': ('7v11', 'OQO'),
    'IAI': ('5sdv', 'IAI'),
    'SAM': ('7c7m', 'SAM'),
}
N_DESIGNS = 8
LENGTH = 150

In [ ]:
def extract_ligand(pdb_id, lig_code):
    arr = bpdb.PDBFile.read(fetch_pdb(pdb_id)).get_structure(model=1)
    mask = arr.res_name == lig_code
    if mask.sum() == 0:
        return None
    first_chain = arr.chain_id[mask][0]
    first_resid = arr.res_id[mask][0]
    sel = mask & (arr.chain_id == first_chain) & (arr.res_id == first_resid)
    lig = arr[sel]
    out = DATA / 'ligands' / f'{pdb_id}_{lig_code}.pdb'
    out.parent.mkdir(parents=True, exist_ok=True)
    f = bpdb.PDBFile(); f.set_structure(lig); f.write(out)
    return out, lig.coord

ligand_data = {}
for name, (pid, code) in LIGANDS.items():
    res = extract_ligand(pid, code)
    if res is None:
        print(f'{name}: ligand {code} not found in {pid}')
        continue
    out, coords = res
    ligand_data[name] = {'pdb': out, 'coords': coords, 'pdb_id': pid, 'code': code}
    print(f'{name}: {len(coords)} atoms -> {out.name}')

## RFdiffusion3 — fixed ligand

In [ ]:
rfd3_lig = DATA / 'rfd3_ligand'
rfd3_lig.mkdir(exist_ok=True)

for name, info in ligand_data.items():
    spec = {
        f'lig_{name}': {
            'input': str(info['pdb']),
            'contig': str(LENGTH),
            'length': f'{LENGTH}-{LENGTH}',
            'ligand': info['code'],
        }
    }
    out_dir = rfd3_lig / name
    ok, err, dt = rfd3_run(spec, out_dir,
                            diffusion_batch_size=N_DESIGNS,
                            num_timesteps=200)
    if not ok:
        print(f'RFD3 {name}: FAILED — {err[-200:]}')
        continue
    append_record(RunRecord(
        model='rfd3', task='sm_binder', target=name, length=LENGTH,
        n_designs=N_DESIGNS, seconds=dt,
        metrics={'s_per_design': dt / N_DESIGNS},
    ))
    print(f'RFD3 {name}: {dt/N_DESIGNS:.1f}s/design')

free_gpu()

## Chroma — `ShapeConditioner` over ligand atom cloud

In [ ]:
from chroma import Chroma, api, conditioners
api.register_key(os.environ['CHROMA_API_KEY'])
chroma = Chroma()

chroma_lig = DATA / 'chroma_ligand'
chroma_lig.mkdir(exist_ok=True)

noise_schedule = chroma.backbone_network.noise_perturb.noise_schedule

for name, info in ligand_data.items():
    times = []
    for i in range(N_DESIGNS):
        try:
            cond = conditioners.ShapeConditioner(
                X_target=info['coords'].astype(np.float32),
                noise_schedule=noise_schedule,
                autoscale_num_residues=LENGTH,
                shape_loss_weight=20.0,
            )
            t0 = time.perf_counter()
            protein = chroma.sample(
                chain_lengths=[LENGTH], steps=200,
                conditioner=cond, sde_func='langevin',
            )
            times.append(time.perf_counter() - t0)
            protein.to(str(chroma_lig / f'{name}_n{i:02d}.pdb'))
        except Exception as e:
            print(f'Chroma {name} #{i}: {type(e).__name__}: {e}')
    if times:
        append_record(RunRecord(
            model='chroma', task='sm_binder', target=name, length=LENGTH,
            n_designs=len(times), seconds=sum(times),
            metrics={'s_per_design': float(np.mean(times))},
        ))
        print(f'Chroma {name}: {np.mean(times):.1f}s/design (n={len(times)})')

free_gpu()

## Pocket-quality metrics

For each design, compute the **buried fraction**: fraction of ligand atoms with
at least one Cα within 6 Å.

In [ ]:
from scipy.spatial import cKDTree

def pocket_metrics(design_pdb, lig_xyz, contact_cut=6.0):
    ca = load_ca_coords(design_pdb)
    if len(ca) == 0 or len(lig_xyz) == 0:
        return None
    tree = cKDTree(ca)
    d, _ = tree.query(lig_xyz, k=1)
    return {
        'buried_frac': float((d < contact_cut).mean()),
        'mean_dist': float(d.mean()),
    }

def find_designs(root, name):
    return sorted(root.rglob(f'*{name}*.pdb'))[:N_DESIGNS]

pocket_summary = {}
for tag, root in [('rfd3', rfd3_lig), ('chroma', chroma_lig)]:
    for name, info in ligand_data.items():
        files = find_designs(root, name)
        if not files: continue
        ms = [m for m in (pocket_metrics(f, info['coords']) for f in files) if m]
        if not ms: continue
        pocket_summary[(tag, name)] = {
            'buried_frac': float(np.mean([m['buried_frac'] for m in ms])),
            'mean_dist':   float(np.mean([m['mean_dist'] for m in ms])),
            'n': len(ms),
        }
        bf = pocket_summary[(tag, name)]
        print(f'{tag:7s} {name}: buried={bf["buried_frac"]:.2f}  d̄={bf["mean_dist"]:.1f}Å  '
              f'(n={bf["n"]})')

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4))
xs = list(LIGANDS.keys())
w = 0.35
for i, model in enumerate(['rfd3', 'chroma']):
    ys = [pocket_summary.get((model, n), {}).get('buried_frac', 0) for n in xs]
    ax.bar(np.arange(len(xs)) + i*w, ys, w, label=model.upper())
ax.set_xticks(np.arange(len(xs)) + w/2); ax.set_xticklabels(xs)
ax.set_ylabel('Mean buried ligand fraction')
ax.set_ylim(0, 1)
ax.set_title('Pocket burial — small molecule')
ax.legend(); ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(RESULTS / 'fig_sm_burial.png', dpi=150)
plt.show()

(RESULTS / 'sm_summary.json').write_text(json.dumps({
    f'{k[0]}_{k[1]}': v for k, v in pocket_summary.items()
}, indent=2))
print('Saved sm_summary.json')